In [4]:
%cd riem-score-metrics/

/home/moritz.burmester/riem-score-metrics


In [8]:
import sys
from pathlib import Path
from PIL import Image
import numpy as np
import torch
import lpips
from experiments_morphbench.sd_wrapper import load_wrapper
import pandas as pd

In [9]:
root = Path('/home/moritz.burmester/riem-score-metrics')
exp = root / 'experiments_morphbench'
sys.path.insert(0, str(root))
sys.path.insert(0, str(exp))

device = 'cuda'
t_level = 0.6

sd = load_wrapper()
loss_fn = lpips.LPIPS(net='vgg').to(device).eval()
print("loaded")

/home/moritz.burmester/anaconda3/envs/id-diff/lib/python3.8/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Scheduler loaded.


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Diffusion pipeline loaded.
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/home/moritz.burmester/anaconda3/envs/id-diff/lib/python3.8/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/moritz.burmester/anaconda3/envs/id-diff/lib/python3.8/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /home/moritz.burmester/anaconda3/envs/id-diff/lib/python3.8/site-packages/lpips/weights/v0.1/vgg.pth
loaded


In [10]:
def to_lpips_tensor(img):
    arr = np.asarray(img.convert('RGB')).astype(np.float32) / 255.0
    t = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
    return (t * 2 - 1).to(device)

@torch.no_grad()
def denoise_curve(pts, emb_cond):
    frames = []
    for i in range(pts.shape[0]):
        zt = pts[i].reshape(1, 4, 64, 64).to(device)
        img = sd.latent2img(sd.latent_backward(zt, emb_cond[i:i+1].to(device),
                                               noise_level=t_level))
        frames.append(img)
    return frames

@torch.no_grad()
def ppl_pdv(frames):
    ds = []
    for a, b in zip(frames[:-1], frames[1:]):
        ds.append(loss_fn(to_lpips_tensor(a), to_lpips_tensor(b)).item())
    ds = np.array(ds)
    ppl = float(ds.sum())
    pdv = float(ds.std())              
    return ppl, pdv, ds

In [11]:
def parse_suffix(stem, d):
    s = stem[len('geodesic_mbm_'):]
    if 'method' in d:                      # lerp / slerp baseline
        return s[:-(len(d['method']) + 1)]
    return s.split('_lam')[0]              # lam metric runs

def method_label(d):
    if 'method' in d:                      # lerp / slerp
        return d['method']
    return f"lam{d['lam']}"

In [15]:
pt_dirs = [exp / 'pt', exp / 'pt_noise']

rows = []
for pt_dir in pt_dirs:
    for f in sorted(pt_dir.glob('geodesic_*.pt')):
        d = torch.load(f, map_location='cpu')
        suffix = parse_suffix(f.stem, d)
        setup_f = pt_dir / f"{suffix}_setup.pt"
        emb_cond = torch.load(setup_f, map_location='cpu')['emb_cond']

        frames = denoise_curve(d['pts'], emb_cond)
        ppl, pdv, ds = ppl_pdv(frames)

        cond = 'z0noise' if d.get('z0noise') in suffix else 'clean'
        rows.append({'name': d['name'], 'method': method_label(d),
                     'condition': cond, 'ppl': ppl, 'pdv': pdv,
                     'adj_lpips': ' '.join(f'{x:.4f}' for x in ds)})
        print(f"{f.stem}: PPL={ppl:.3f} PDV={pdv:.3f}")

df = pd.DataFrame(rows)
df.to_csv(exp / 'ppl_pdv.csv', index=False)
print(f"\n{len(df)} curves")

  0%|                                                                                                                            | 0/30 [00:00<?, ?it/s]

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30/30 [00:03<00:00,  7.74it/s]


geodesic_mbm_anakin_lam00_it10: PPL=1.438 PDV=0.061


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30/30 [00:03<00:00,  7.73it/s]


geodesic_mbm_anakin_lam00_it500: PPL=1.453 PDV=0.077


...

geodesic_mbm_girl_lam10_it500: PPL=0.567 PDV=0.025


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30/30 [00:03<00:00,  7.68it/s]


geodesic_mbm_girl_lerp: PPL=0.352 PDV=0.026


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30/30 [00:03<00:00,  7.66it/s]


KeyboardInterrupt: 